In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EMP04 - BRIBERY & CORRUPTION INVESTIGATIONS
   ===================================================================================
   BUSINESS QUESTION:
   How many distinct employee investigation cases tied to bribery, corruption, or
   related code of conduct violations were opened during the assessment period,
   by Assessable Unit?

   DATA SOURCES:
      - hive_metastore.ra_adido_2025.replace_with_emp04_investigation_table
      - vw_cost_center_mapping_bootstrap
      - hive_metastore.ra_adido_2025.fy25_list_of_units

   IMPLEMENTATION NOTES:
      1. Replace the source table and placeholder column names before running.
      2. Confirm the metric grain is CASE-level and keep COUNT(DISTINCT Case_ID).
      3. Confirm the assessment-period date column. The scaffold uses Open_Date.
      4. If the source file is actually a closed-case extract, align the business
         question and the date/status filter before using the result.

   QUERY LOGIC & SUMMARY:
      1. MASTER AU ANCHOR: Restrict final output to the standard ABAC AU list
         (Canada, Hong Kong, Barbados plus US_OR_CANADA = 'CANADA').
      2. CASE IDENTIFICATION: Flag only cases that meet the agreed bribery /
         corruption indicator rule using the designated source columns.
      3. ASSESSMENT PERIOD RULE: Keep only cases opened during the assessment
         period using the agreed case-open date column.
      4. COST CENTER NORMALIZATION: Normalize the source-side Cost Center using
         the shared canonical rule before joining to the mapping bootstrap view.
      5. AU BRIDGE: Join normalized Cost Center to vw_cost_center_mapping_bootstrap
         and roll cases up to AU_ID.
      6. AGGREGATION: Count DISTINCT Case_ID per AU.
      7. FALLBACK: LEFT JOIN the counts back to Master_AUs so all in-scope AUs
         return a row and default missing counts to 0.
=================================================================================== */

WITH Master_AUs AS (
    SELECT
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    SELECT DISTINCT
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE
            WHEN TRIM(CAST(Cost_Center_ID AS STRING)) RLIKE '^[0-9]+$'
            THEN LPAD(RIGHT(TRIM(CAST(Cost_Center_ID AS STRING)), 4), 4, '0')
            ELSE UPPER(TRIM(CAST(Cost_Center_ID AS STRING)))
        END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
      AND Cost_Center_ID IS NOT NULL
),

Investigation_Source AS (
    SELECT
        TRIM(CAST(REPLACE_WITH_CASE_ID AS STRING)) AS Case_ID,
        CASE
            WHEN TRIM(CAST(REPLACE_WITH_COST_CENTER AS STRING)) RLIKE '^[0-9]+$'
            THEN LPAD(RIGHT(TRIM(CAST(REPLACE_WITH_COST_CENTER AS STRING)), 4), 4, '0')
            ELSE UPPER(TRIM(CAST(REPLACE_WITH_COST_CENTER AS STRING)))
        END AS Cleaned_CC,
        UPPER(TRIM(CAST(REPLACE_WITH_TARGET_FLAG_COL_G AS STRING))) AS Target_Flag_G,
        UPPER(TRIM(CAST(REPLACE_WITH_TARGET_FLAG_COL_I AS STRING))) AS Target_Flag_I,
        UPPER(TRIM(CAST(REPLACE_WITH_CASE_STATUS AS STRING))) AS Case_Status,
        TO_DATE(REPLACE_WITH_OPEN_DATE) AS Open_Date
    FROM hive_metastore.ra_adido_2025.replace_with_emp04_investigation_table
    WHERE REPLACE_WITH_CASE_ID IS NOT NULL
      AND REPLACE_WITH_COST_CENTER IS NOT NULL
),

Flagged_Cases AS (
    SELECT DISTINCT
        Case_ID,
        Cleaned_CC
    FROM Investigation_Source
    WHERE Open_Date BETWEEN DATE '2024-11-01' AND DATE '2025-10-31'
      AND (
            Target_Flag_G IN ('Y', 'YES')
         OR Target_Flag_I IN ('Y', 'YES')
      )
      AND COALESCE(Case_Status, '') NOT IN ('CANCELLED', 'VOID')
),

Bridged_Cases AS (
    SELECT DISTINCT
        m.AU_ID,
        f.Case_ID,
        f.Cleaned_CC
    FROM Flagged_Cases f
    INNER JOIN CC_Mapping m
        ON f.Cleaned_CC = m.Mapped_CC
),

AU_Case_Counts AS (
    SELECT
        AU_ID,
        COUNT(DISTINCT Case_ID) AS Flagged_Case_Count
    FROM Bridged_Cases
    GROUP BY AU_ID
)

SELECT
    a.BusinessID,
    a.AU_Name,
    a.Business_Segment,
    'EMP04' AS QuestionID,
    COALESCE(CAST(c.Flagged_Case_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN AU_Case_Counts c
    ON a.BusinessID = c.AU_ID
ORDER BY a.BusinessID;


In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EMP04 - AU Level Case Routing Review
   PURPOSE: One row per AU showing normalized Cost Centers, distinct mapped case
   counts, and why the result is zero or non-zero.
=================================================================================== */

WITH Master_AUs AS (
    SELECT
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    SELECT DISTINCT
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE
            WHEN TRIM(CAST(Cost_Center_ID AS STRING)) RLIKE '^[0-9]+$'
            THEN LPAD(RIGHT(TRIM(CAST(Cost_Center_ID AS STRING)), 4), 4, '0')
            ELSE UPPER(TRIM(CAST(Cost_Center_ID AS STRING)))
        END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
      AND Cost_Center_ID IS NOT NULL
),

Investigation_Source AS (
    SELECT
        TRIM(CAST(REPLACE_WITH_CASE_ID AS STRING)) AS Case_ID,
        CASE
            WHEN TRIM(CAST(REPLACE_WITH_COST_CENTER AS STRING)) RLIKE '^[0-9]+$'
            THEN LPAD(RIGHT(TRIM(CAST(REPLACE_WITH_COST_CENTER AS STRING)), 4), 4, '0')
            ELSE UPPER(TRIM(CAST(REPLACE_WITH_COST_CENTER AS STRING)))
        END AS Cleaned_CC,
        UPPER(TRIM(CAST(REPLACE_WITH_TARGET_FLAG_COL_G AS STRING))) AS Target_Flag_G,
        UPPER(TRIM(CAST(REPLACE_WITH_TARGET_FLAG_COL_I AS STRING))) AS Target_Flag_I,
        UPPER(TRIM(CAST(REPLACE_WITH_CASE_STATUS AS STRING))) AS Case_Status,
        TO_DATE(REPLACE_WITH_OPEN_DATE) AS Open_Date
    FROM hive_metastore.ra_adido_2025.replace_with_emp04_investigation_table
    WHERE REPLACE_WITH_CASE_ID IS NOT NULL
      AND REPLACE_WITH_COST_CENTER IS NOT NULL
),

Flagged_Cases AS (
    SELECT DISTINCT
        Case_ID,
        Cleaned_CC
    FROM Investigation_Source
    WHERE Open_Date BETWEEN DATE '2024-11-01' AND DATE '2025-10-31'
      AND (
            Target_Flag_G IN ('Y', 'YES')
         OR Target_Flag_I IN ('Y', 'YES')
      )
      AND COALESCE(Case_Status, '') NOT IN ('CANCELLED', 'VOID')
),

Bridged_Cases AS (
    SELECT DISTINCT
        m.AU_ID,
        f.Case_ID,
        f.Cleaned_CC
    FROM Flagged_Cases f
    INNER JOIN CC_Mapping m
        ON f.Cleaned_CC = m.Mapped_CC
),

AU_Case_Counts AS (
    SELECT
        AU_ID,
        COUNT(DISTINCT Case_ID) AS Included_Case_Count
    FROM Bridged_Cases
    GROUP BY AU_ID
),

AU_Cost_Center_List AS (
    SELECT
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Normalized_Cost_Centers
    FROM Bridged_Cases
    GROUP BY AU_ID
)

-- Output columns:
-- BusinessID: Master AU ID from the ABAC AU anchor list.
-- AU_Name: Master AU name tied to BusinessID.
-- Business_Segment: Business segment carried from the master AU list.
-- QuestionID: Questionnaire code for this debug output.
-- Response: Final case count returned for EMP04.
-- Included_Case_Count: Distinct mapped flagged cases included in the result.
-- Normalized_Cost_Centers: Normalized cost centers on counted cases for this AU.
-- Debug_Reason: Plain-English explanation of why the result is zero or non-zero.
-- In_ABAC_AU_List: Confirms the AU came from the standard ABAC AU list.
SELECT
    a.BusinessID,
    a.AU_Name,
    a.Business_Segment,
    'EMP04' AS QuestionID,
    COALESCE(CAST(c.Included_Case_Count AS STRING), '0') AS Response,
    COALESCE(c.Included_Case_Count, 0) AS Included_Case_Count,
    COALESCE(cc.Normalized_Cost_Centers, '') AS Normalized_Cost_Centers,
    CASE
        WHEN COALESCE(c.Included_Case_Count, 0) > 0 THEN 'Mapped flagged cases found in the assessment period.'
        ELSE 'No mapped flagged cases found for this AU in the assessment period.'
    END AS Debug_Reason,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN AU_Case_Counts c
    ON a.BusinessID = c.AU_ID
LEFT JOIN AU_Cost_Center_List cc
    ON a.BusinessID = cc.AU_ID
ORDER BY a.BusinessID;
